In [1]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Linear models
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier, Perceptron

# SVM
from sklearn.svm import SVC, LinearSVC

# Neighbors
from sklearn.neighbors import KNeighborsClassifier

# Trees & ensembles
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    BaggingClassifier
)

# Probabilistic
from sklearn.naive_bayes import GaussianNB

# Discriminant analysis
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis

# Metrics
from sklearn.metrics import accuracy_score, f1_score

from util.data_fft import *

# =========================================================
# 1. DATAFRAME
# =========================================================
data_fft = pd.read_csv('data/itsc_generators_edf_v1.0.csv')

data_diff = data_fft.drop(data_fft.columns[-1:], axis=1)

df = pd.concat(
    [data_diff, data_fft['Machine']],
    axis=1
)

# =========================================================
# 2. COLUMNS
# =========================================================
feature_cols = [
    "3.3 Hz", "6.7 Hz", "10 Hz", "13.3 Hz", "16.7 Hz",
    "20 Hz", "23.3 Hz", "26.7 Hz", "30.0 Hz", "33.3 Hz",
    "36.7 Hz", "40 Hz", "43.3 Hz", "46.7 Hz"
]

target_col = "y"
group_col = "Machine"

# =========================================================
# 3. SPLITS
# =========================================================
splits = [
    (['D', 'B', 'C', 'F', 'A'], 'E'),
    (['E', 'B', 'C', 'F', 'A'], 'D'),
    (['E', 'D', 'C', 'F', 'A'], 'B'),
    (['E', 'D', 'B', 'F', 'A'], 'C'),
    (['E', 'D', 'B', 'C', 'A'], 'F'),
    (['E', 'D', 'B', 'C', 'F'], 'A')
]

# =========================================================
# 4. MODELS (EXTENDED)
# =========================================================
models = {

    # -------------------------
    # Linear models
    # -------------------------
    "LogReg": Pipeline([
        ("clf", LogisticRegression(max_iter=1000, class_weight='balanced'))
    ]),

    "Ridge": Pipeline([
        ("clf", RidgeClassifier(class_weight='balanced'))
    ]),

    # -------------------------
    # SVM
    # -------------------------
    "SVM_rbf": Pipeline([
        ("clf", SVC(kernel="rbf", class_weight='balanced'))
    ]),

    "SVM_linear": Pipeline([
        ("clf", LinearSVC(max_iter=5000, class_weight='balanced'))
    ]),

    # -------------------------
    # Neighbors
    # -------------------------
    "KNN": Pipeline([
        ("clf", KNeighborsClassifier(n_neighbors=5))
    ]),

    # -------------------------
    # Trees
    # -------------------------
    "DecisionTree": DecisionTreeClassifier(random_state=42, class_weight='balanced'),

    # -------------------------
    # Ensembles
    # -------------------------
    "RandomForest": RandomForestClassifier(n_estimators=10, random_state=42, class_weight='balanced'),

    "ExtraTrees": ExtraTreesClassifier(n_estimators=10, random_state=42, class_weight='balanced'),

    "GradientBoosting": GradientBoostingClassifier(),

    "AdaBoost": AdaBoostClassifier(),

    "Bagging": BaggingClassifier(),

    # -------------------------
    # Probabilistic
    # -------------------------
    "NaiveBayes": GaussianNB(),

    # -------------------------
    # Discriminant Analysis
    # -------------------------
    "LDA": LinearDiscriminantAnalysis(),
}

# =========================================================
# 5. TRAIN / TEST
# =========================================================
results = []

for i, (train_machines, test_machine) in enumerate(splits):

    df_train = df[df[group_col].isin(train_machines)]
    df_test  = df[df[group_col] == test_machine]

    X_train = df_train[feature_cols].values
    y_train = df_train[target_col].values

    X_test = df_test[feature_cols].values
    y_test = df_test[target_col].values

    split_result = {"Split": f"S{i+1}"}

    for name, model in models.items():
        try:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            f1  = f1_score(y_test, y_pred, average="weighted")

        except Exception as e:
            # Handle rare numerical issues (e.g., QDA singular matrix)
           f1 = np.nan, np.nan

        split_result[f"{name}_f1"]  = f1

    results.append(split_result)

# =========================================================
# 6. RESULTS
# =========================================================
df_results = pd.DataFrame(results)

mean_row = df_results.drop(columns=["Split"]).mean()
std_row  = df_results.drop(columns=["Split"]).std()

mean_row["Split"] = "Mean"
std_row["Split"]  = "Std"

df_final = pd.concat(
    [df_results, pd.DataFrame([mean_row, std_row])],
    ignore_index=True
)

cols = ["Split"] + [c for c in df_final.columns if c != "Split"]
df_final = df_final[cols]

# =========================================================
# 7. LATEX EXPORT
# =========================================================
latex_table = df_final.to_latex(
    index=False,
    float_format="%.4f",
    column_format="l" + "c" * (len(df_final.columns) - 1),
    escape=True 
)

print("\n===== LATEX TABLE =====\n")
print(latex_table)

/Users/morvan/Documents/Recherche/Publications/XAI_ITSC_2026/itsc-xai-generators/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



===== LATEX TABLE =====

\begin{tabular}{lccccccccccccc}
\toprule
Split & LogReg\_f1 & Ridge\_f1 & SVM\_rbf\_f1 & SVM\_linear\_f1 & KNN\_f1 & DecisionTree\_f1 & RandomForest\_f1 & ExtraTrees\_f1 & GradientBoosting\_f1 & AdaBoost\_f1 & Bagging\_f1 & NaiveBayes\_f1 & LDA\_f1 \\
\midrule
S1 & 0.9655 & 0.8462 & 0.6364 & 0.8462 & 0.2353 & 0.6957 & 0.4211 & 0.6957 & 0.9655 & 0.8462 & 0.2353 & 0.8000 & 0.6364 \\
S2 & 0.9655 & 0.9655 & 1.0000 & 0.9655 & 0.5714 & 0.9655 & 1.0000 & 0.9286 & 0.8000 & 1.0000 & 0.9655 & 1.0000 & 1.0000 \\
S3 & 1.0000 & 1.0000 & 0.9655 & 1.0000 & 0.9286 & 0.8462 & 0.9655 & 1.0000 & 0.8462 & 0.8462 & 0.8462 & 0.7500 & 0.9286 \\
S4 & 0.0000 & 0.0000 & 0.1250 & 0.0000 & 0.2353 & 0.0000 & 0.6957 & 0.4211 & 0.1250 & 0.5000 & 0.0000 & 0.1250 & 0.5000 \\
S5 & 0.0000 & 0.0000 & 0.0000 & 0.0000 & 0.0000 & 0.0000 & 0.0000 & 0.0000 & 0.0000 & 0.0000 & 0.0000 & 0.0000 & 0.2353 \\
S6 & 0.0000 & 0.0000 & 0.0000 & 0.0000 & 0.0000 & 0.0000 & 0.0000 & 0.0000 & 0.0000 & 0.0000 & 0.0

In [2]:
display(df_final)

,Split,LogReg_f1,Ridge_f1,SVM_rbf_f1,SVM_linear_f1,KNN_f1,DecisionTree_f1,RandomForest_f1,ExtraTrees_f1,GradientBoosting_f1,AdaBoost_f1,Bagging_f1,NaiveBayes_f1,LDA_f1
0,S1,0.965517,0.846154,0.636364,0.846154,0.235294,0.695652,0.421053,0.695652,0.965517,0.846154,0.235294,0.800000,0.636364
1,S2,0.965517,0.965517,1.000000,0.965517,0.571429,0.965517,1.000000,0.928571,0.800000,1.000000,0.965517,1.000000,1.000000
2,S3,1.000000,1.000000,0.965517,1.000000,0.928571,0.846154,0.965517,1.000000,0.846154,0.846154,0.846154,0.750000,0.928571
3,S4,0.000000,0.000000,0.125000,0.000000,0.235294,0.000000,0.695652,0.421053,0.125000,0.500000,0.000000,0.125000,0.500000
4,S5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.235294
5,S6,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,Mean,0.488506,0.468612,0.454480,0.468612,0.328431,0.417887,0.513704,0.507546,0.456112,0.532051,0.341161,0.445833,0.550038
7,Std,0.535279,0.515872,0.471910,0.515872,0.361192,0.465694,0.449465,0.442232,0.459477,0.443505,0.448379,0.452884,0.389182
